## <span style="color: Violet;"> **Импорт библиотек и Интерфейсы** </style> 

In [7]:
import json
import os
import copy
from abc import ABC, abstractmethod
from typing import List, Optional, Dict, Any

class IUnit(ABC):
    @property
    @abstractmethod
    def name(self) -> str: pass
    
    @property
    @abstractmethod
    def hp(self) -> int: pass
    
    @abstractmethod
    def take_damage(self, amount: int) -> None: pass

class ILandscape(ABC):
    @property
    @abstractmethod
    def symbol(self) -> str: pass

    @property
    @abstractmethod
    def move_cost(self) -> int: pass

class INeutralObject(ABC):
    @property
    @abstractmethod
    def symbol(self) -> str: pass

    @abstractmethod
    def effect(self, unit: 'IUnit') -> bool:
        """Возвращает True, если объект поглощается/уничтожается после взаимодействия"""
        pass


## <span style="color: Plum;"> **Окружение и Нейтральные объекты** </style> 

In [8]:
# Landscapes

class Plain(ILandscape):
    symbol = '.'
    move_cost = 1

class Forest(ILandscape):
    symbol = '🌲'
    move_cost = 2

class Mountain(ILandscape):
    symbol = "🗻"
    move_cost = 99

class Reaver(ILandscape):
    symbol = '_'
    move_cost = 4

# Neutral Objects

class HealingShrine(INeutralObject):
    symbol = '⛩️'
    def effect(self, unit: IUnit) -> bool:
        hp_before = unit.hp
        unit.hp = min(unit.hp + 20, unit.max_hp)
        hp_restored = unit.hp - hp_before
        print(f"Древнее божество услышало молитвы {unit.name} ")
        print(f"{unit.name} восстановил {hp_restored} HP")

class Rill(INeutralObject):
    symbol = '='
    def effect(self, unit: IUnit) -> bool:
        hp_before = unit.hp
        unit.hp = min(unit.hp + 5, unit.max_hp)
        hp_restored = unit.hp - hp_before
        print(f"{unit.name} выпил водички из родника")
        print(f"{unit.name} восстановил {hp_restored} HP")

class Chest(INeutralObject):
    symbol = '❌'
    def effect(self, unit: IUnit) -> bool:
        print(f"{unit.name} обнаружил сундук, однако удача отвернулась от него: внутри было пусто")
        return True

class Trap(INeutralObject):
    symbol = '_'
    def effect(self, unit: IUnit) -> bool:
        unit.take_damage(20)
        print(f"{unit.name} угодил в ловушку")
        print(f"{unit.name} потерял 20 HP")
        return True

class EnrichedMine(INeutralObject):
    symbol = '⛏️'
    def effect(self, unit: IUnit) -> bool:
        print(f"{unit.name} успешно извлёк ценные ресурсы из недр обогащённого рудника")
        return False


## <span style="color: purple;"> **Юниты** </style> 

In [9]:
class Unit(IUnit):
    def __init__(self, name: str, hp: int, attack: int, armor: int, move_range: int):
        self._name = name
        self.max_hp = hp
        self._hp = hp
        self.attack = attack
        self.armor = armor
        self.move_range = move_range
        self.x = 0
        self.y = 0

        @property
        def name(self) -> str: return self._name

        @property
        def hp(self) -> int: return self._hp

        @hp.setter
        def hp(self, value: int): self._hp = max(0, value)

        def take_damage(self, amount: int) -> None:
            damage = max(0, amount - self.armor)
            self.hp -= damage

        def __add__(self, other: INeutralObject):
            if isinstance(other, INeutralObject):
                is_consumed = other.effect(self)
                return is_consumed
            raise TypeError("Юнит может взаимодействовать с INeutralObject только через оператор «+»")
# Единицы ближнего боя
class Zealot(Unit):
    def __init__(self): super().__init__("Зилот", hp=100, attack = 15, armor = 15, move_range = 3)
class DarkTemplar(Unit):
    def __init__(self): super().__init__("Тёмный тамплиер", hp=50, attack = 30, armor = 5, move_range = 4)
# Единицы дальнего боя
class Dragoon(Unit):
    def __init__(self): super().__init__("Драгун", hp=120, attack = 20, armor = 20, move_range = 3)
class Stalker(Unit):
    def __init__(self): super().__init__("Сталкер", hp=100, attack = 15, armor = 10, move_range = 5)
# Тяжёлые механические единицы
class Reaver(Unit):
    def __init__(self): super().__init__("Опустошитель", hp=100, attack = 80, armor = 5, move_range = 1)
class Immortal(Unit):
    def __init__(self): super().__init__("Бессмертный", hp=150, attack = 20, armor = 40, move_range = 2)
class Colossus(Unit):
    def __init__(self): super().__init__("Колосс", hp=200, attack = 40, armor = 20, move_range = 8)


## <span style="color: #957B8D;"> **Игровое поле и База** </style> 

In [10]:
import copy
from typing import List, Optional

class Cell:
    """Клетка игрового поля — содержит ландшафт, юнита, нейтральный объект или базу"""
    def __init__(self):
        self.landscape: ILandscape = Plain()
        self.unit: Optional[IUnit] = None
        self.neutral_object: Optional[INeutralObject] = None
        self.base: Optional['GameBase'] = None

class GameBase:
    """База игрока — создаёт юнитов и учитывает их"""
    def __init__(self, x: int, y: int, max_units: int = 5):
        self.x = x
        self.y = y
        self.hp = 500
        self.max_units = max_units
        self.created_units: List[IUnit] = []
        self.gold = 1000 

    def create_unit(self, unit_class) -> Optional[IUnit]:
        """Создание юнита на базе с проверкой лимита"""
        if len(self.created_units) >= self.max_units:
            print(f"Достигнут лимит юнитов на базе! Максимум: {self.max_units}")
            return None

        unit = unit_class()


        if hasattr(unit, 'cost') and unit.cost > self.gold:
            print(f"Недостаточно минералов! Нужно {unit.cost}, у вас {self.gold}")
            return None

        if hasattr(unit, 'cost'):
            self.gold -= unit.cost

        unit.x, unit.y = self.x, self.y
        self.created_units.append(unit)
        print(f"Создан юнит: {unit.name} (HP: {unit.hp}, Атака: {unit.attack}, Броня: {unit.armor})")
        return unit

    def remove_unit(self, unit: IUnit):
        """Удаление юнита из списка созданных"""
        if unit in self.created_units:
            self.created_units.remove(unit)
            print(f"Юнит {unit.name} удалён из списка базы")

    def get_status(self) -> str:
        """Получение статуса базы в виде строки"""
        return (f"База на ({self.x}, {self.y}) | HP: {self.hp} | "
                f"Юнитов: {len(self.created_units)}/{self.max_units} | "
                f"Минералы: {self.gold}")

class GameField:
    """Игровое поле — контейнер для всех объектов"""
    
    def __init__(self, width: int, height: int, max_objects: int = 50):
        self.width = width
        self.height = height
        self.max_objects = max_objects
        self.grid: List[List[Cell]] = [[Cell() for _ in range(width)] for _ in range(height)]
        self.bases: List[GameBase] = []
        self._object_count = 0

    def set_landscape(self, x: int, y: int, landscape: ILandscape):
        """Установка типа ландшафта на клетке"""
        if 0 <= x < self.width and 0 <= y < self.height:
            self.grid[y][x].landscape = landscape

    def add_unit(self, unit: IUnit, x: int, y: int) -> bool:
        """Добавление юнита на поле с контролем лимита объектов"""
        if self._object_count >= self.max_objects:
            print(f"Достигнут лимит объектов на поле! Максимум: {self.max_objects}")
            return False
        
        if not (0 <= x < self.width and 0 <= y < self.height):
            print("Координаты вне поля!")
            return False
            
        if self.grid[y][x].unit is not None:
            print("Клетка уже занята другим юнитом!")
            return False
            
        self.grid[y][x].unit = unit
        unit.x, unit.y = x, y
        self._object_count += 1
        return True

    def remove_unit(self, unit: IUnit):
        """Удаление юнита с поля"""
        if 0 <= unit.y < self.height and 0 <= unit.x < self.width:
            if self.grid[unit.y][unit.x].unit == unit:
                self.grid[unit.y][unit.x].unit = None
                self._object_count -= 1

    def add_base(self, base: GameBase):
        """Размещение базы на поле"""
        self.bases.append(base)
        self.grid[base.y][base.x].base = base
        self._object_count += 1

    def move_unit(self, unit: IUnit, new_x: int, new_y: int) -> bool:
        """Перемещение юнита с учётом ландшафта и лимитов"""
        if not (0 <= new_x < self.width and 0 <= new_y < self.height):
            print("Целевая клетка находится за пределами поля!")
            return False
        
        target_cell = self.grid[new_y][new_x]
        
        if target_cell.unit is not None:
            print("Клетка занята другим юнитом!")
            return False
            
        if target_cell.base is not None:
            print("Нельзя встать на базу!")
            return False

        # Проверка дальности хода
        distance = abs(new_x - unit.x) + abs(new_y - unit.y)
        if distance > unit.move_range:
            print(f"Слишком далеко! Дальность хода юнита: {unit.move_range}, расстояние: {distance}")
            return False
        
        # Проверка проходимости ландшафта
        if target_cell.landscape.movement_cost > unit.move_range:
            print(f"Местность непроходима! Стоимость: {target_cell.landscape.movement_cost}")
            return False

        # Перемещение юнита
        self.grid[unit.y][unit.x].unit = None
        unit.x, unit.y = new_x, new_y
        target_cell.unit = unit
        
        # Взаимодействие с нейтральным объектом через перегрузку оператора +
        if target_cell.neutral_object is not None:
            is_consumed = unit + target_cell.neutral_object
            if is_consumed:
                target_cell.neutral_object = None
                self._object_count -= 1
                print(f"Нейтральный объект на клетке ({new_x}, {new_y}) использован")
                
        return True

    def add_neutral_object(self, x: int, y: int, obj: INeutralObject):
        """Добавление нейтрального объекта на поле"""
        if not (0 <= x < self.width and 0 <= y < self.height):
            print("Координаты вне поля!")
            return False
            
        if self.grid[y][x].neutral_object is not None:
            print("На этой клетке уже есть нейтральный объект!")
            return False
            
        self.grid[y][x].neutral_object = obj
        self._object_count += 1
        return True
    
    def copy_field(self) -> 'GameField':
        """Глубокое копирование поля со всеми объектами"""
        return copy.deepcopy(self)

    def render(self) -> str:
        """Отрисовка поля в виде строки"""
        # Заголовок с координатами X
        result = "   " + "".join(f"{i % 10} " for i in range(self.width)) + "\n"
        result += "  +" + "--" * self.width + "+\n"
        
        for y in range(self.height):
            row_str = f"{y % 10} |"
            for x in range(self.width):
                cell = self.grid[y][x]
                if cell.base:
                    row_str += "B "
                elif cell.unit:
                    name = cell.unit.name
                    if "Зилот" in name:
                        row_str += "Z "
                    elif "Тёмный тамплиер" in name:
                        row_str += "D "
                    elif "Драгун" in name:
                        row_str += "G "
                    elif "Сталкер" in name:
                        row_str += "S "
                    elif "Опустошитель" in name:
                        row_str += "R "
                    elif "Бессмертный" in name:
                        row_str += "I "
                    elif "Колосс" in name:
                        row_str += "C "
                    else:
                        row_str += "U "
                elif cell.neutral_object:
                    row_str += cell.neutral_object.symbol + " "
                else:
                    row_str += cell.landscape.symbol + " "
            result += row_str + "|\n"
        
        result += "  +" + "--" * self.width + "+"
        return result


## <span style="color: Lavender;"> **Задача 19(Работа с JSON)** </style> 

In [11]:
import os
import json
from typing import Any
class JsonManager:
    "Инициализация файла"
    def __init__ (self, filepath, str):
        self.filepath = filepath
        if not os.path.exists(self.filepath):
            self.write()

    "Функция для чтения файла"
    def read(self) -> dict:
        try:
            with open(self.filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        except (FileNotFoundError, json.JSONDecodeError):
            return {}
            
    "Функция для полной перезаписи файла"
    def write(self, data: dict) -> None:
        if not isinstance(data, dict):
            raise TypeError("Корень JSON должен быть словарём (dict) для корректной работы методов.")
        with open(self.filepath, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)

    "Функция для добавление новой пары ключ-значение"
    def add(self, key: str, value: Any) -> None:
        data = self.read()
        if key in data:
            raise ValueError(f"Ключ {key} уже существует, используйте update().")
        data[key]=value
        self.write(data)

    "Функция для обновления ключа" 
    def update(self, key: str, value: Any) -> None:
        data = self.read()
        if key not in data:
            raise KeyError(f"Ключ {key} не найден")
        data[key] = value
        self.write(data)

    "Функция для удаления ключа"
    def remove(self, key: str, value: Any) -> None:
            data = self.read()
            if key not in data:
                raise KeyError(f"Ключ {key} не найден")
            del data[key]
            self.write(data)
    "Функция для получения ключа"
    def get(self, key: str, default=None):
        return self.read().get(key, default)